# SatQuery AI — Real Point Cloud RAG Test

**Goal of this notebook:** prove out one thing end to end — real Indian point-cloud data → tiled, rasterized images → real CLIP embeddings → real cosine-similarity retrieval against a plain-language query. Nothing here is scripted; every number in the results cell is computed.

**Run in Google Colab** (Runtime → Change runtime type → GPU, though CPU works fine too for one village's worth of data).

**This is a test, not the integration.** The goal today is: does this pipeline actually work on this actual data. Once it does, the last cell exports everything needed to wire it into the app — that's a separate, later step.

**Scope, on purpose:** one village, tiled into a handful of images, indexed and queried. Not fine-tuning, not a new model — a pretrained CLIP encoder doing real inference, same as the rest of SatQuery's pipeline.

## 0. One-time setup: Kaggle credentials in Colab Secrets

Do this once, ever — it persists across every future Colab session, no re-upload needed:

1. Go to kaggle.com → Settings → API → **Create New Token**. This downloads `kaggle.json`.
2. Open that file. It contains `{"username": "...", "key": "..."}`.
3. In this Colab notebook, click the **🔑 key icon** in the left sidebar (Secrets).
4. Add two secrets: `KAGGLE_USERNAME` (the username value) and `KAGGLE_KEY` (the key value).
5. Toggle **Notebook access** on for both.

That's it — every cell below reads them from Secrets. You never touch `kaggle.json` again after this.

In [ ]:
# Install dependencies. PLY/PCD reading is hand-rolled with numpy below (see
# section 3) rather than via open3d, which frequently lacks a wheel for
# whatever Python version Colab is currently running.
!pip install -q kagglehub laspy open_clip_torch pillow numpy


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception as e:
    print("Could not load from Colab Secrets:", e)
    print("Falling back: if you have kaggle.json locally, upload it now via the")
    print("file browser and uncomment the two lines below.")
    # import shutil, pathlib
    # pathlib.Path("/root/.kaggle").mkdir(exist_ok=True)
    # shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")


## 1. Download the dataset (no manual upload)

In [ ]:
import kagglehub

DATASET = "jaideepchouhan/10-indian-villages-point-cloud-data"
dataset_path = kagglehub.dataset_download(DATASET)
print("Downloaded to:", dataset_path)


## 2. Inspect what's actually in it

I could not see this dataset's real file listing ahead of time (Kaggle needs auth to browse), so this cell is deliberately a discovery step, not an assumption. It will tell you what formats and files actually exist before anything downstream tries to load them.

In [ ]:
import os
from collections import defaultdict

POINT_CLOUD_EXTS = {".las", ".laz", ".ply", ".pcd", ".xyz", ".txt", ".csv", ".pts", ".e57"}

all_files = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        full = os.path.join(root, f)
        all_files.append(full)

print(f"Total files found: {len(all_files)}\n")

by_ext = defaultdict(list)
for f in all_files:
    ext = os.path.splitext(f)[1].lower()
    by_ext[ext].append(f)

print("By extension:")
for ext, files in sorted(by_ext.items(), key=lambda x: -len(x[1])):
    sizes = [os.path.getsize(f) for f in files]
    print(f"  {ext or '(no ext)'}: {len(files)} files, "
          f"{sum(sizes)/1e6:.1f} MB total, "
          f"{max(sizes)/1e6:.1f} MB largest")

point_cloud_files = [f for f in all_files if os.path.splitext(f)[1].lower() in POINT_CLOUD_EXTS]
print(f"\nLikely point-cloud files: {len(point_cloud_files)}")
for f in point_cloud_files[:20]:
    print(" ", os.path.relpath(f, dataset_path), f"({os.path.getsize(f)/1e6:.1f} MB)")
if len(point_cloud_files) > 20:
    print(f"  ... and {len(point_cloud_files)-20} more")


## 3. Pick one village and load it

Set `VILLAGE_FILE` below once you've seen the listing above — defaults to the first point-cloud file found, so this runs out of the box on a first pass. This loader handles `.ply` and `.pcd` with a small hand-rolled numpy parser (no open3d dependency), `.las`/`.laz` via `laspy`, and falls back to plain numeric text for anything else.

In [ ]:
import numpy as np
import struct

VILLAGE_FILE = point_cloud_files[0] if point_cloud_files else None
assert VILLAGE_FILE, "No point-cloud-like file found -- check the extension list above and adjust POINT_CLOUD_EXTS."
print("Loading:", VILLAGE_FILE)

# NOTE: open3d is deliberately not used here. As of late 2026 it frequently has
# no published wheel for the newest Python (Colab is on 3.13), which produces
# exactly the "Could not find a version that satisfies the requirement" error.
# These two readers cover PLY and PCD -- by far the most common point-cloud
# export formats from photogrammetry pipelines (Metashape, Pix4D, ODM,
# COLMAP) -- using only numpy, which always has a wheel for whatever Python
# Colab happens to be running.

PLY_TYPE_MAP = {
    "float": "f4", "float32": "f4", "double": "f8", "float64": "f8",
    "uchar": "u1", "uint8": "u1", "char": "i1", "int8": "i1",
    "short": "i2", "int16": "i2", "ushort": "u2", "uint16": "u2",
    "int": "i4", "int32": "i4", "uint": "u4", "uint32": "u4",
}

def read_ply(path):
    """Handles ASCII and binary_little_endian/big_endian PLY, the formats
    every common photogrammetry tool actually exports."""
    with open(path, "rb") as f:
        header_lines = []
        while True:
            line = f.readline().decode("ascii", errors="replace").strip()
            header_lines.append(line)
            if line == "end_header":
                break

        fmt = "ascii"
        if "binary_little_endian" in header_lines[1]:
            fmt = "binary_little_endian"
        elif "binary_big_endian" in header_lines[1]:
            fmt = "binary_big_endian"

        n_vertices, properties, prop_types = 0, [], {}
        in_vertex = False
        for line in header_lines:
            if line.startswith("element vertex"):
                n_vertices = int(line.split()[-1])
                in_vertex = True
            elif line.startswith("element"):
                in_vertex = False
            elif line.startswith("property") and in_vertex:
                parts = line.split()
                properties.append(parts[-1])
                prop_types[parts[-1]] = parts[1]

        if fmt == "ascii":
            # np.loadtxt reads straight into a numpy array without building a huge
            # Python list first -- the previous list-comprehension approach could
            # exhaust RAM and crash the kernel on a multi-million-point file.
            data = np.loadtxt(f, max_rows=n_vertices, dtype=np.float64)
            col = {p: i for i, p in enumerate(properties)}
            xyz = data[:, [col["x"], col["y"], col["z"]]]
            rgb = None
            if all(k in col for k in ("red", "green", "blue")):
                rgb = data[:, [col["red"], col["green"], col["blue"]]].astype(np.uint8)
        else:
            endian = "<" if fmt == "binary_little_endian" else ">"
            dtype = np.dtype([(p, endian + PLY_TYPE_MAP.get(prop_types[p], "f4")) for p in properties])
            data = np.fromfile(f, dtype=dtype, count=n_vertices)
            xyz = np.vstack([data["x"], data["y"], data["z"]]).T.astype(np.float64)
            rgb = None
            if all(k in data.dtype.names for k in ("red", "green", "blue")):
                rgb = np.vstack([data["red"], data["green"], data["blue"]]).T.astype(np.uint8)
        return xyz, rgb


def read_pcd(path):
    """Handles ASCII PCD. Binary PCD is comparatively rare for shared
    datasets; if you hit one, check whether the same data is also offered
    as .ply or .las in the listing above -- that is the faster fix."""
    with open(path, "rb") as f:
        fields, n_points, data_fmt = [], 0, "ascii"
        while True:
            raw = f.readline()
            line = raw.decode("ascii", errors="replace").strip()
            if not line:
                continue
            key = line.split()[0]
            if key == "FIELDS":
                fields = line.split()[1:]
            elif key == "POINTS":
                n_points = int(line.split()[1])
            elif key == "DATA":
                data_fmt = line.split()[1]
                break
        if data_fmt != "ascii":
            raise NotImplementedError(
                "This fallback PCD reader only handles ASCII PCD. "
                "Check the file listing above for a .ply or .las version of the same data instead."
            )
        # Same fix as read_ply: numpy's own reader instead of a Python list loop.
        data = np.loadtxt(f, max_rows=n_points, dtype=np.float64)
        col = {name: i for i, name in enumerate(fields)}
        xyz = data[:, [col["x"], col["y"], col["z"]]]
        rgb = None
        if "rgb" in col:
            packed = data[:, col["rgb"]].astype(np.float32).view(np.int32)
            r = (packed >> 16) & 0xFF
            g = (packed >> 8) & 0xFF
            b = packed & 0xFF
            rgb = np.vstack([r, g, b]).T.astype(np.uint8)
        elif all(k in col for k in ("r", "g", "b")):
            rgb = data[:, [col["r"], col["g"], col["b"]]].astype(np.uint8)
        return xyz, rgb


def load_point_cloud(path):
    """Returns (xyz: (N,3) float array, rgb: (N,3) uint8 array or None)."""
    ext = os.path.splitext(path)[1].lower()

    if ext == ".ply":
        return read_ply(path)

    if ext == ".pcd":
        return read_pcd(path)

    if ext in (".las", ".laz"):
        import laspy
        las = laspy.read(path)
        xyz = np.vstack([las.x, las.y, las.z]).T
        rgb = None
        if hasattr(las, "red"):
            r, g, b = las.red, las.green, las.blue
            if r.max() > 255:
                r, g, b = (r / 256).astype(np.uint8), (g / 256).astype(np.uint8), (b / 256).astype(np.uint8)
            rgb = np.vstack([r, g, b]).T.astype(np.uint8)
        return xyz, rgb

    # Last resort: plain-text point list, x y z [r g b ...].
    try:
        arr = np.loadtxt(path, delimiter=None)
    except Exception:
        arr = np.loadtxt(path, delimiter=",")
    xyz = arr[:, :3]
    rgb = arr[:, 3:6].astype(np.uint8) if arr.shape[1] >= 6 else None
    return xyz, rgb

xyz, rgb = load_point_cloud(VILLAGE_FILE)

print(f"\nPoints: {len(xyz):,}")
print(f"Has real RGB colour: {rgb is not None}")
print(f"Extent X: {xyz[:,0].min():.2f} to {xyz[:,0].max():.2f}  ({xyz[:,0].max()-xyz[:,0].min():.1f} units across)")
print(f"Extent Y: {xyz[:,1].min():.2f} to {xyz[:,1].max():.2f}  ({xyz[:,1].max()-xyz[:,1].min():.1f} units across)")
print(f"Extent Z: {xyz[:,2].min():.2f} to {xyz[:,2].max():.2f}  ({xyz[:,2].max()-xyz[:,2].min():.1f} units of relief)")
if rgb is not None:
    print(f"RGB sample (first 3 points): {rgb[:3].tolist()}")


## 4. Tile the village and rasterize each tile

Top-down orthographic projection, one image per tile. If real RGB exists, each pixel takes the colour of the **highest** point in that column (what you'd actually see looking straight down) — not an averaged blend of ground and canopy. Falls back to an elevation colour ramp if there's no RGB.

In [ ]:
from PIL import Image
import matplotlib.cm as cm

GRID_N = 3          # 3x3 = 9 tiles; drop to 2 for a sparser cloud, raise for a denser one
IMG_PX = 512
MIN_POINTS_PER_TILE = 500   # tiles with fewer points than this are skipped as too sparse to be useful

x0, x1 = xyz[:,0].min(), xyz[:,0].max()
y0, y1 = xyz[:,1].min(), xyz[:,1].max()
z0, z1 = xyz[:,2].min(), xyz[:,2].max()

def rasterize_tile(txyz, trgb, px=IMG_PX):
    """Top-down raster of one tile. Returns a PIL Image."""
    tx0, tx1 = txyz[:,0].min(), txyz[:,0].max()
    ty0, ty1 = txyz[:,1].min(), txyz[:,1].max()
    span = max(tx1 - tx0, ty1 - ty0, 1e-6)

    col = (((txyz[:,0] - tx0) / span) * (px - 1)).astype(int)
    row = ((1 - (txyz[:,1] - ty0) / span) * (px - 1)).astype(int)  # flip Y so north is up
    col = np.clip(col, 0, px - 1)
    row = np.clip(row, 0, px - 1)

    z_buffer = np.full((px, px), -np.inf)
    img = np.zeros((px, px, 3), dtype=np.uint8)
    occupied = np.zeros((px, px), dtype=bool)

    if trgb is not None:
        colours = trgb
    else:
        z_norm = np.clip((txyz[:,2] - z0) / max(z1 - z0, 1e-6), 0, 1)
        colours = (cm.terrain(z_norm)[:, :3] * 255).astype(np.uint8)

    # Highest point wins per pixel -- a real top-down view, not a blended average.
    order = np.argsort(txyz[:,2])  # ascending, so later writes (higher Z) overwrite
    for i in order:
        r, c = row[i], col[i]
        if txyz[i,2] >= z_buffer[r, c]:
            z_buffer[r, c] = txyz[i,2]
            img[r, c] = colours[i]
            occupied[r, c] = True

    # Light fill for empty pixels so sparse tiles don't look full of black holes.
    if occupied.mean() < 0.9:
        from scipy.ndimage import distance_transform_edt
        idx = distance_transform_edt(~occupied, return_distances=False, return_indices=True)
        img = img[tuple(idx)]

    return Image.fromarray(img)

tiles = []
xs = np.linspace(x0, x1, GRID_N + 1)
ys = np.linspace(y0, y1, GRID_N + 1)

for i in range(GRID_N):
    for j in range(GRID_N):
        mask = (xyz[:,0] >= xs[i]) & (xyz[:,0] < xs[i+1]) & (xyz[:,1] >= ys[j]) & (xyz[:,1] < ys[j+1])
        n = mask.sum()
        if n < MIN_POINTS_PER_TILE:
            continue
        timg = rasterize_tile(xyz[mask], rgb[mask] if rgb is not None else None)
        tiles.append({
            "id": f"tile_{i}_{j}",
            "image": timg,
            "n_points": int(n),
            "bbox": [float(xs[i]), float(ys[j]), float(xs[i+1]), float(ys[j+1])],
            "mean_elev": float(xyz[mask,2].mean()),
        })

print(f"Rasterized {len(tiles)} tiles from {GRID_N*GRID_N} candidate grid cells "
      f"({GRID_N*GRID_N - len(tiles)} skipped as too sparse).")
for t in tiles:
    print(f"  {t['id']}: {t['n_points']:,} pts, mean elev {t['mean_elev']:.1f}")


In [ ]:
# Quick visual sanity check before spending time embedding anything.
import matplotlib.pyplot as plt

n_show = min(len(tiles), 9)
fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
if n_show == 1:
    axes = [axes]
for ax, t in zip(axes, tiles[:n_show]):
    ax.imshow(t["image"])
    ax.set_title(t["id"], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Embed every tile with a pretrained CLIP encoder

Same model family already used elsewhere in this project. No training, no fine-tuning — real inference on real images.

In [ ]:
import torch, open_clip
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(DEVICE).eval()
print("Encoder loaded on", DEVICE)

@torch.no_grad()
def embed_image(pil_img):
    t = preprocess(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)
    e = model.encode_image(t)
    return F.normalize(e, dim=-1).cpu()

@torch.no_grad()
def embed_text(text):
    tok = tokenizer([text]).to(DEVICE)
    e = model.encode_text(tok)
    return F.normalize(e, dim=-1).cpu()

for t in tiles:
    t["embedding"] = embed_image(t["image"])

print(f"Embedded {len(tiles)} tiles, {tiles[0]['embedding'].shape[-1]}-dim vectors.")


## 6. The retrieval function — this is the actual RAG step

Query text → real text embedding → real cosine similarity against every tile → ranked results. Nothing precomputed or faked here; every score below is computed at call time, same as `CONTRACT.md` INV-1 requires for the main app.

In [ ]:
def query(text, top_k=3):
    q = embed_text(text)
    scored = []
    for t in tiles:
        sim = float((q @ t["embedding"].T).item())
        scored.append((sim, t))
    scored.sort(key=lambda x: -x[0])
    return scored[:top_k]

TEST_QUERIES = [
    "dense cluster of buildings",
    "open ground with no structures",
    "vegetation and trees",
    "a road or path",
]

for q in TEST_QUERIES:
    results = query(q, top_k=1)
    sim, t = results[0]
    print(f'"{q}"  ->  {t["id"]}  (confidence {sim:.4f}, {t["n_points"]:,} pts)')


In [ ]:
# Visual check: does the top result for each query actually look right?
fig, axes = plt.subplots(1, len(TEST_QUERIES), figsize=(3.5*len(TEST_QUERIES), 3.5))
for ax, q in zip(axes, TEST_QUERIES):
    sim, t = query(q, top_k=1)[0]
    ax.imshow(t["image"])
    ax.set_title(f'"{q}"\n{t["id"]} · {sim:.3f}', fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("\nIf these matches look wrong, that's real signal, not a bug to hide:")
print("general-purpose CLIP was never trained on top-down point-cloud rasters,")
print("so weak matches here are honest information about a real limitation --")
print("worth knowing before demo day, not after.")


## 7. Try your own queries

In [ ]:
MY_QUERY = "show me the area with the most structures"  # edit and re-run

for sim, t in query(MY_QUERY, top_k=3):
    print(f"{t['id']}: confidence {sim:.4f}  |  bbox {t['bbox']}  |  {t['n_points']:,} pts")

best = query(MY_QUERY, top_k=1)[0][1]
plt.figure(figsize=(4,4))
plt.imshow(best["image"])
plt.title(f'"{MY_QUERY}" -> {best["id"]}')
plt.axis("off")
plt.show()


## 8. Export for later integration

Not wired into the app yet — that's a separate step once this pipeline is confirmed working on the real data. This just packages what a future integration step would need: the tile images, their real embeddings, and metadata.

In [ ]:
import json, zipfile

OUT_DIR = "/content/satquery_village_export"
os.makedirs(OUT_DIR, exist_ok=True)

manifest = []
for t in tiles:
    img_path = os.path.join(OUT_DIR, f'{t["id"]}.jpg')
    t["image"].convert("RGB").save(img_path, quality=88)
    manifest.append({
        "id": t["id"],
        "image_file": f'{t["id"]}.jpg',
        "n_points": t["n_points"],
        "bbox": t["bbox"],
        "mean_elev": t["mean_elev"],
        "embedding": t["embedding"].squeeze(0).tolist(),
    })

manifest_path = os.path.join(OUT_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump({
        "source_file": os.path.relpath(VILLAGE_FILE, dataset_path),
        "encoder": "open_clip ViT-B-32 (openai pretrained)",
        "tiles": manifest,
    }, f, indent=1)

zip_path = "/content/satquery_village_export.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, OUT_DIR))

print("Exported:", zip_path)
print(f"Contains {len(manifest)} tiles + manifest.json (images + real embeddings + bboxes).")
print("Download this file from the Colab file browser once you're happy with the results above.")


## 9. Export a demo-ready manifest for the app (curated queries, real scores)

Everything above proves the pipeline works with fully open-ended queries. For tonight's app integration, the safer shape is a **small curated set of realistic queries with real precomputed results** — the app displays these instantly with zero live inference, zero network dependency at demo time, and zero new risk to the working flood scenario.

This is the same pattern `CONTRACT.md` already uses for the flood/quake/infra scenarios: fixed corpus, real numbers. The only difference is these numbers come from a real CLIP embedding of a real point cloud instead of a text-only lexical match.

Edit `CANDIDATE_QUERIES` below to match what you actually want to be able to ask live tomorrow.

In [ ]:
import base64, io

CANDIDATE_QUERIES = [
    "show me the area with the most buildings",
    "where is the open, undeveloped ground",
    "which area has the most vegetation",
    "is there a road or clear path through this village",
]

def image_to_data_uri(pil_img, quality=82):
    buf = io.BytesIO()
    pil_img.convert("RGB").save(buf, format="JPEG", quality=quality)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/jpeg;base64,{b64}"

demo_entries = []
for q in CANDIDATE_QUERIES:
    sim, t = query(q, top_k=1)[0]
    demo_entries.append({
        "query": q,
        "tile_id": t["id"],
        "confidence": round(sim, 4),
        "n_points": t["n_points"],
        "bbox": t["bbox"],
        "image_url": image_to_data_uri(t["image"]),
    })
    print(f'{q!r:55} -> {t["id"]}  confidence={sim:.4f}')

manifest = {
    "scenario_id": "village",
    "scenario_name": "Real Indian Village — Point Cloud Retrieval",
    "source_file": os.path.relpath(VILLAGE_FILE, dataset_path),
    "encoder": "open_clip ViT-B-32 (openai pretrained), zero-shot, no fine-tuning",
    "is_real_data": True,
    "entries": demo_entries,
}

manifest_path = "/content/village_demo_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=1)

size_kb = os.path.getsize(manifest_path) / 1024
print(f"\nWrote {manifest_path}  ({size_kb:.0f} KB, {len(demo_entries)} curated queries, images embedded as base64)")
print("Download this one file from the Colab file browser — that's the only artifact the app integration needs.")
